# Alpamayo-Edge — 一次性导出/量化 (Colab 免费版)

产出:`Alpamayo-R1-10B` FP16 ONNX + `Cosmos-Reason2-8B` INT4/INT8,打包下载后 scp 到 Orin。

> ⚠️ **显存/内存提醒(诚实)**:Colab 免费是 **T4 16GB + ~13GB RAM**。
> - **Cosmos-Reason2-8B INT4/INT8 量化**:偏紧但通常可行(必要时用 device_map offload)。
> - **Alpamayo-R1-10B FP16 导出**:~20GB,**免费 T4 很可能 OOM**。建议:Runtime→Change runtime type 选 **High-RAM**;
>   仍不行就退回 **Kaggle 2×T4=32GB**。可先只跑 Cosmos(第 3 节),Alpamayo(第 4 节)另想办法。


## 1. 检查环境


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | awk '/Mem:/{print "RAM total",$2"GB"}'


## 2. 安装 TensorRT-Edge-LLM + 登录 HF
(先在 HF 网站同意 `nvidia/Alpamayo-R1-10B` 许可)


In [ ]:
!git clone --depth 1 https://github.com/NVIDIA/TensorRT-Edge-LLM.git
%cd TensorRT-Edge-LLM
!pip -q install ".[tools]"
import os; os.environ['EDGE_LLM_PATH']=os.getcwd(); os.environ['PYTHONPATH']=os.getcwd()
%cd /content
from huggingface_hub import notebook_login; notebook_login()   # 贴 HF token


## 3. Cosmos-Reason2-8B → INT4 / INT8 (量化轨道,先跑这个)


In [ ]:
import os
for QF in ['int4_awq','int8_sq']:
    out=f'Cosmos-Reason2-8B-{QF}'
    os.system(f'tensorrt-edgellm-quantize llm --model_dir nvidia/Cosmos-Reason2-8B --output_dir {out} --qformat {QF}')
    os.system(f'tensorrt-edgellm-export {out} {out}/onnx')
print('cosmos done')


## 4. Alpamayo-R1-10B → FP16 ONNX (VLA 轨道;显存不足会 OOM,见顶部提示)


In [ ]:
!hf download nvidia/Alpamayo-R1-10B --local-dir Alpamayo-R1-10B
!tensorrt-edgellm-export Alpamayo-R1-10B Alpamayo-R1-10B/onnx --max-kv-cache-capacity 4096


## 5. 打包 ONNX 下载 (再 scp 到 Orin ~/tensorrt-edgellm-workspace/)


In [ ]:
!tar -czf edge_artifacts.tgz Alpamayo-R1-10B/onnx Cosmos-Reason2-8B-int4_awq/onnx Cosmos-Reason2-8B-int8_sq/onnx
from google.colab import files; files.download('edge_artifacts.tgz')
